# 🔺 Delta Lake — Comportamento Básico, _delta_log e File Skipping

## Objetivos
- Entender o comportamento básico do Delta Lake
- Compreender a estrutura e o significado do `_delta_log`
- Entender a importância do File Skipping (pulo de arquivos)

---

## 1. Setup

Desativamos as otimizações automáticas do Databricks para observar o comportamento puro do Delta Lake.

In [ ]:
%%sql
-- Desativa o autoOtimize para não mascarar o comportamento do Delta Lake
SET spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite = false;
SET spark.databricks.delta.properties.defaults.autoOptimize.autoCompact = false;

---
## 2. Definição da Tabela

Criamos a tabela `demo` com uma constraint `NOT NULL` e uma `CHECK` para validação de dados.

In [ ]:
%%sql
DROP TABLE IF EXISTS demo;

CREATE TABLE demo (
  id    INT,
  valid BOOLEAN,
  name  STRING
);

In [ ]:
%%sql
-- Coluna valid não pode ser nula
ALTER TABLE demo ALTER COLUMN valid SET NOT NULL;

-- id deve estar entre 1 e 999998
ALTER TABLE demo ADD CONSTRAINT Ids CHECK (id > 0 AND id < 999999);

In [ ]:
%%sql
DESCRIBE demo;

In [ ]:
%%sql
-- Mostra localização, formato, propriedades e constraints
DESCRIBE EXTENDED demo;

---
## 3. Explorando o _delta_log (tabela vazia)

Mesmo sem dados, o Delta Lake já cria a estrutura de log com arquivos `.json` e `.crc`.

In [ ]:
%fs
ls dbfs:/user/hive/warehouse/demo

In [ ]:
%fs
ls dbfs:/user/hive/warehouse/demo/_delta_log/

### Estrutura do `_delta_log`

| Arquivo | Descrição |
|---|---|
| `.json` | Commit file — registra cada operação transacional (a fonte da verdade) |
| `.crc` | Cyclic Redundancy Check — checksum para verificar integridade do `.json` |
| `.checkpoint.parquet` | Compacta N jsons anteriores em um único arquivo para leitura mais rápida |
| `.compacted.json` | Compactação similar ao checkpoint, porém em formato JSON |

In [ ]:
%%sql
-- Lê o primeiro commit (criação da tabela)
-- Colunas retornadas: commitInfo (metadados da operação) e metaData (schema/formato/config Delta)
SELECT * FROM json.`dbfs:/user/hive/warehouse/demo/_delta_log/00000000000000000000.json`

In [ ]:
%%sql
-- Visualiza o histórico de transações
DESC HISTORY demo;

---
## 4. Inserindo Dados

### 4.1 Insert — 1 linha válida

In [ ]:
%%sql
INSERT INTO demo VALUES (1, True, 'Alice');

In [ ]:
-- Um novo arquivo .snappy.parquet é criado com os dados físicos
%fs
ls dbfs:/user/hive/warehouse/demo

In [ ]:
-- Um novo .json é adicionado ao _delta_log registrando o commit
%fs
ls dbfs:/user/hive/warehouse/demo/_delta_log/

### 4.2 Insert com violação de constraint

> **Conceito ACID — Atomicidade**  
> O Delta Lake implementa transações usando *optimistic concurrency* e gravação em duas etapas:  
> 1. Cria o arquivo `.parquet` físico  
> 2. Valida constraints e, se tudo estiver OK, grava o `.json` no `_delta_log` (commit)  
>
> Sem o `.json` correspondente, o `.parquet` é considerado **órfão** e é completamente **ignorado** pela tabela.

In [ ]:
%%sql
-- ❌ Viola a constraint: id deve ser > 0
-- Retorna erro, mas pode gerar um .parquet órfão no diretório
INSERT INTO demo VALUES (-1, True, 'Martin');

In [ ]:
-- Pode aparecer um novo .parquet, mas...
%fs
ls dbfs:/user/hive/warehouse/demo

In [ ]:
-- ...NÃO há novo .json no _delta_log → transação não foi commitada
%fs
ls dbfs:/user/hive/warehouse/demo/_delta_log/

In [ ]:
%%sql
-- O histórico também não registra a operação inválida
DESC HISTORY demo;

### 4.3 Insert — múltiplas linhas

Múltiplos `VALUES` em um único `INSERT` geram **apenas 1 arquivo `.parquet` e 1 commit `.json`**.

In [ ]:
%%sql
INSERT INTO demo VALUES
  (2, False, 'Bob'),
  (3, False, 'Steve'),
  (4, False, 'Ray');

In [ ]:
%%sql
-- Lê o commit gerado pelo insert múltiplo
-- Colunas: 'add' (metadados + STATS do arquivo) e 'commitInfo'
-- stats contém: numRecords, minValues, maxValues, nullCount
-- ⚠️ Por padrão, o Databricks coleta estatísticas apenas das primeiras 32 colunas
SELECT * FROM json.`dbfs:/user/hive/warehouse/demo/_delta_log/00000000000000000004.json`

---
## 5. Update

> **Como o Delta Lake trata Updates?**  
> Ao contrário de bancos relacionais tradicionais (que modificam o arquivo físico existente),  
> o Delta Lake **cria um novo arquivo `.parquet`** com os dados atualizados.  
> O arquivo antigo é marcado como `remove` no `_delta_log`.  
>  
> Vantagens:  
> ✅ Mais rápido (sem random write no mesmo arquivo)  
> ✅ Habilita **Time Travel** (equivalente ao Flashback do Oracle)

In [ ]:
%%sql
SELECT * FROM demo;

In [ ]:
%%sql
UPDATE demo
SET name = 'Alice Updated'
WHERE name = 'Alice';

In [ ]:
%%sql
SELECT * FROM demo;

In [ ]:
%%sql
-- Commit do UPDATE contém 3 colunas:
-- 'add'        → novo .parquet com dados atualizados
-- 'remove'     → .parquet antigo marcado como removido (ainda existe fisicamente)
-- 'commitInfo' → metadados da operação
SELECT * FROM json.`dbfs:/user/hive/warehouse/demo/_delta_log/00000000000000000005.json`

---
## 6. File Skipping (Pulo de Arquivos)

> O Delta Lake é chamado de **single source of truth** porque o Spark **nunca** lê os `.parquet` diretamente.  
> Primeiro, ele consulta o `_delta_log` para saber:  
> - Quais arquivos estão ativos (`add`)  
> - Quais foram removidos (`remove`)  
> - As **estatísticas** de cada arquivo (`minValues`, `maxValues`)  
>  
> Isso permite **pular arquivos** que certamente não contêm os dados do filtro.  
> 📌 **Como verificar:** Spark UI → SQL/DataFrame → "Scan parquet" → número de arquivos lidos

In [ ]:
%%sql
-- Sem filtro → lê TODOS os arquivos ativos (ex: 2 arquivos na Spark UI)
SELECT * FROM demo;

In [ ]:
%%sql
-- Com filtro → Spark usa as stats do _delta_log para identificar
-- que 'Alice Updated' só pode estar em 1 arquivo → lê apenas 1 arquivo
SELECT * FROM demo WHERE name = 'Alice Updated';

---
## 7. Delete

> No Delta Lake, um `DELETE` não cria um novo `.parquet` imediatamente.  
> Em vez disso, é gerado um **Deletion Vector** (arquivo `.bin`), que marca as linhas deletadas  
> de forma leve e eficiente — sem reescrever o arquivo inteiro.

In [ ]:
%%sql
DELETE FROM demo WHERE name = 'Alice Updated';

In [ ]:
-- Nenhum novo .parquet criado — apenas um arquivo deletion_vector....bin
%fs
ls dbfs:/user/hive/warehouse/demo

---
## 8. Merge (UPSERT)

O `MERGE` permite combinar INSERT, UPDATE e DELETE em uma única operação atômica.

In [ ]:
%%sql
-- Tabela auxiliar com as atualizações a aplicar
CREATE OR REPLACE TABLE demo_updates (
  id        INT,
  valid     BOOLEAN,
  name      STRING,
  operation STRING
);

INSERT INTO demo_updates VALUES
  (4, True,  'Ray',  ''),        -- update (matched, sem operação especial)
  (5, True,  'Miki', ''),        -- insert (not matched)
  (2, False, 'Bob',  'delete');  -- delete (matched com operação 'delete')

In [ ]:
%%sql
SELECT * FROM demo_updates;

In [ ]:
%%sql
MERGE INTO demo
USING demo_updates
ON demo.id = demo_updates.id
WHEN MATCHED AND demo_updates.operation = 'delete' THEN
    DELETE
WHEN MATCHED THEN
    UPDATE SET
        demo.valid = demo_updates.valid,
        demo.name  = demo_updates.name
WHEN NOT MATCHED THEN
    INSERT (id, valid, name)
    VALUES (demo_updates.id, demo_updates.valid, demo_updates.name);

In [ ]:
%%sql
-- id=2 (Bob) deletado, id=4 (Ray) atualizado, id=5 (Miki) inserido
SELECT * FROM demo;

---
## 9. Checkpoint

> À medida que a tabela sofre muitas operações, acumula-se um grande número de arquivos `.json` no `_delta_log`.  
> Ler centenas de JSONs antes de cada query seria muito lento.  
>  
> Para resolver isso, o Delta Lake cria automaticamente **checkpoints**:  
>
> | Arquivo | Descrição |
> |---|---|
> | `.checkpoint.parquet` | Resume o estado completo da tabela até aquele ponto. Substitui a leitura de todos os JSONs anteriores. Criado a cada 10 commits por padrão. |
> | `.compacted.json` | Compactação similar, porém mantém o formato JSON. Menos comum. |
> | `_last_checkpoint` | Arquivo de metadados que aponta para o checkpoint mais recente. |

In [ ]:
%%sql
-- Após 10 commits, um checkpoint é criado automaticamente
-- Você pode forçar a criação com:
-- OPTIMIZE demo;  (também compacta os arquivos pequenos)

DESC HISTORY demo;

In [ ]:
-- Verifica o _delta_log após muitas operações (pode conter .checkpoint.parquet)
%fs
ls dbfs:/user/hive/warehouse/demo/_delta_log/

---
## 10. Resumo — Fluxo Interno do Delta Lake

```
┌─────────────────────────────────────────────────────────────┐
│                     DELTA LAKE — FLUXO                      │
│                                                             │
│  WRITE (INSERT/UPDATE/DELETE/MERGE)                         │
│    1. Grava arquivo .parquet com os dados novos             │
│    2. Valida constraints e regras                           │
│    3. Se OK  → cria .json no _delta_log (COMMIT)            │
│       Se FAIL → .parquet fica órfão, SEM commit             │
│                                                             │
│  READ (SELECT)                                              │
│    1. Lê _delta_log para listar arquivos ativos             │
│    2. Usa stats (min/max) → File Skipping                   │
│    3. Lê apenas os .parquet necessários                     │
│                                                             │
│  MANUTENÇÃO                                                 │
│    • A cada 10 commits → gera .checkpoint.parquet           │
│    • VACUUM remove .parquet órfãos e versões antigas        │
│    • OPTIMIZE compacta arquivos pequenos (bin packing)      │
└─────────────────────────────────────────────────────────────┘
```

### Propriedades ACID garantidas pelo Delta Lake

| Propriedade | Como o Delta Lake implementa |
|---|---|
| **Atomicidade** | Commit só ocorre com o `.json` no `_delta_log`; sem `.json` = rollback |
| **Consistência** | Constraints (`NOT NULL`, `CHECK`) validadas antes do commit |
| **Isolamento** | Optimistic concurrency control nos commits |
| **Durabilidade** | `.crc` garante integridade dos arquivos de log |